# 第2回：データを見て、問いを立て、評価を正しく設計する

この回は3つのパートで構成します：**データ探偵—分布・欠損・外れ値 ／ 何を、いつ、何のために予測するか ／ モデルは本当に当たっているか**。

**セルの動かし方**：各セル（灰色の枠）を選んで `Shift + Enter`（またはセル左の▷ボタン）を押すと実行できます。
**上から順に**実行してください。前のセルを飛ばすと、後のセルでエラーになります。

**AIと一緒に進める**：分からないコードは、セル全体ではなく気になる数行をM365 CopilotなどのAIへ貼って
説明や修正を相談します（`ASK COPILOT`）。ただしAIの答えは鵜呑みにせず、必ず自分の出力で確かめます。

`TRY`は全員、`CHANGE`は値を1つ変える練習、`CHALLENGE`は余裕がある人向け、
`DEEP DIVE`・`APPENDIX`は発展です（飛ばしても本編は完結します）。


In [ ]:
# 【準備セル】教材フォルダの場所を自動で見つけます。中身は今は理解しなくてOK、そのまま実行してください。
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回でできるようになること

データの怪しい点を見つけ（EDA）、何をいつ予測するかを決め（問題設定）、その評価がどこまで信じられるか（検証とリーク）を設計します。

- 単変量・二変量・群別の順でデータを見る
- 欠損の発生機構と外れ値を、検定や多変量手法で客観的に調べる
- 図と統計量から、断定ではなく検証可能な仮説を作る
- 利用者・判断・予測時点を1文にする
- 目的変数と利用可能な説明変数を分け、リーク候補を自動監査する
- 誤りのコストから期待値を計算し、指標と閾値を業務要件で決める
- 学習・検証・テストの役割を区別し、前処理を分割の内側へ入れる
- 分割方式ごとのスコアのばらつきを比べ、楽観的な評価を見抜く
- ネストした交差検証とadversarial validationで、楽観の少ない推定と分布ずれを確かめる

### この回の進み方（大切）

この回は、旧カリキュラムの**3回分をまとめた長い回**です。**パート1→2→3**の順に、各パートの
`CORE`（本線）で手を動かします。1回の時間で全部を終える必要はありません。各パートの
`DEEP DIVE`／`APPENDIX`は、余裕のある人や自習で進めてください。日をまたいで少しずつでも大丈夫です。

### 先に押さえる言葉

- 分布：値がどこにどれだけ存在するか
- 外れ値：他と大きく異なる観測値
- 相互情報量：非線形も捉える関連の強さ
- 欠損機構：MCAR/MAR/MNARという欠損の起こり方
- 多変量外れ値：単変量では見えない組み合わせの異常
- 予測時点：モデルを実際に使う瞬間
- ベースライン：複雑なモデルと比較する単純な基準
- コスト行列：誤りの種類ごとの損失をまとめた表
- 期待コスト：確率×損失で見積もる平均的な損失
- リーク監査：目的変数と強く結びつく怪しい列を洗い出す確認
- 汎化：未知データでも性能を保つこと
- リーク：予測時には得られない情報が学習へ混ざること
- グループ分割：関連試料を同じ側へまとめる分割
- ネストCV：探索と評価を二重の交差検証で分ける方法
- adversarial validation：学習とテストを見分けられるか調べる手法

> **実行前の30秒予想**：各パートの問いに、今の言葉で仮の答えを書いてから始めます。


---

# パート1：データ探偵—分布・欠損・外れ値

**このパートの問い：モデルを作る前に、データの怪しいところをどう見つけるか。**


## EDA＝モデルを作る前にデータをよく見る工程

EDA（探索的データ分析）は、いきなりモデルを作らず、まずデータをよく見る工程です。目的は
「きれいなグラフを作ること」ではなく、**モデルを惑わせる怪しい点（偏り・欠損・外れ値）を先に見つけ、
検証できる仮説を作ること**です。

見る順番にはコツがあります：**1変数（分布）→ 2変数（関係）→ 群別（カテゴリごと）**。
いきなり複雑な図に行かず、単純な図から積み上げます。次のセルはまず描画の下準備（日本語表示と
見た目のテーマ設定）です。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


In [ ]:
# グラフの日本語が文字化けしないようにする設定です。中身は今は理解しなくてOK、そのまま実行してください。
import matplotlib.pyplot as plt
from matplotlib import font_manager
for _name in ["Yu Gothic", "Meiryo", "Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic"]:
    if _name in {f.name for f in font_manager.fontManager.ttflist}:
        plt.rcParams["font.family"] = _name
        break
import seaborn as sns
sns.set_theme(style="whitegrid")


## TRY：1変数の分布を見る（ヒストグラムと箱ひげ図）

まず1列ずつ「値がどこに、どれだけあるか」を見ます。

- **ヒストグラム**：値を区間に分け、各区間の件数を棒で表す。山の形・偏り・飛び離れた値が見えます。
- **箱ひげ図**：中央値・四分位・外れ値候補（ひげの外の点）をコンパクトに表す。外れ値探しに向きます。


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=df, x="yield_pct", bins=20, ax=axes[0])
axes[0].set_title("収率の分布")
sns.boxplot(data=df, x="reaction_time_h", ax=axes[1])
axes[1].set_title("反応時間：外れ値候補を探す")
plt.tight_layout()


### 出力の読み方

- **左（収率のヒストグラム）**：山が1つか2つか、左右どちらに裾を引くかを見ます。裾が長い＝一部に極端な値。
- **右（反応時間の箱ひげ図）**：箱が中央50%、ひげの外の点が外れ値候補。**右端にぽつんと離れた点**があれば、それが要調査の試料です（このデータには意図的に極端な値を仕込んであります）。
- まだ「削除」はしません。EDAは**見つける**段階です。


## 2変数の関係とカテゴリ比較（散布図・箱ひげ図）

次に「2つの列の関係」を見ます。散布図は連続値どうしの関係、色分け（`hue`）で3つ目の情報（触媒）も
重ねられます。カテゴリごとの違いは箱ひげ図で比べます。


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.scatterplot(data=df, x="temperature_c", y="yield_pct", hue="catalyst", alpha=0.65, ax=axes[0])
axes[0].set_title("温度と収率")
sns.boxplot(data=df, x="catalyst", y="yield_pct", ax=axes[1])
axes[1].set_title("触媒別の収率")
plt.tight_layout()


### 出力の読み方

- **左（温度×収率）**：右肩上がりの直線ではなく、**中くらいの温度で収率が高くなる山型**に見えるはずです。「関係＝直線」とは限らないことを、目で確認しておきます（第4回DEEP DIVEの相互情報量につながります）。
- 点の色（触媒）で**かたまり**ができていれば、触媒が収率に効いている手がかり。
- **右（触媒別の箱ひげ）**：触媒ごとに箱の高さ（収率の中心）が違えば、触媒の効果が疑われます。ただし件数が少ない触媒は割り引いて読みます。


## TRY：欠損と「明らかに怪しい値」を表で押さえる

図で当たりを付けたら、表で具体的に特定します。どの列にいくつ欠損があるか、そして温度が異常に
大きい上位5件を実際に取り出します。


In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].to_frame("欠損数"))
display(df.nlargest(5, "temperature_c")[["sample_id", "temperature_c", "reaction_time_h", "yield_pct"]])


### 出力の読み方

- **1つ目の表**：欠損のある列と件数。欠損の多い列は、後で「埋める／落とす／別扱い」の判断が要ります。
- **2つ目の表**：温度が高い順の5件。`180.0`のような**周囲から突出した値**があれば、入力ミスか特殊な実験かを疑い、`sample_id`を控えて確認先を考えます。
- ここでも即削除しないのが鉄則。**「誰に確認するか」「残した場合に何が起きるか」**まで考えてから対処します。


## CHANGE

散布図の色分け（`hue`）を`catalyst`から`solvent`へ変え、見え方の違いを1つ挙げます。

## 注意

外れ値＝入力ミス、ではありません。本物の珍しい現象のこともあります。EDAの結論は「削除」ではなく、
**「確認すべき仮説」**の形で残します。


## DEEP DIVE：印象を統計量で裏づける

図の印象は主観的です。ここでは4つの道具で客観化します：**相関**（直線的な関係）、
**相互情報量**（曲がった関係も拾う）、**欠損機構**（欠損の起こり方）、**多変量外れ値**（組み合わせの異常）。


### 相関ヒートマップ：全列の関係を一望する

`corr()`は数値列すべての**相関係数**（-1〜+1）を計算します。+1に近いほど一緒に増え、-1に近いほど
片方が増えると片方が減る関係。色の濃淡で一望できます。ただし**相関は「直線的な」関係しか測れない**
ことに注意します。


In [ ]:
numeric_cols = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa", "yield_pct"]
correlation = df[numeric_cols].corr()
plt.figure(figsize=(8, 5))
sns.heatmap(correlation, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("数値列の相関（因果ではない）")
plt.tight_layout()


### 出力の読み方

- 対角線は自分自身との相関で必ず1.00。赤いマスほど正、青いマスほど負の相関。
- `temperature_c`と`yield_pct`の相関は**意外と弱い**はずです（山型の関係なので直線相関では捉えきれない）。
- **重要**：相関は因果ではありません。「AとBが一緒に動く」ことと「AがBの原因」は別物です。


### 相互情報量：曲がった関係も拾う

温度のように「最適点で収率が最大」の山型は、相関では弱く見えます。**相互情報量**は直線に限らず
「片方を知るともう片方の予想がどれだけ絞れるか」を測るので、こうした関係を拾えます。相関と並べて読みます。


In [ ]:
from sklearn.feature_selection import mutual_info_regression

mi_source = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa"]
mi_frame = df[mi_source + ["yield_pct"]].dropna()
mi = mutual_info_regression(mi_frame[mi_source], mi_frame["yield_pct"], random_state=42)
pearson = mi_frame[mi_source].corrwith(mi_frame["yield_pct"]).abs()
compare = pd.DataFrame({"相互情報量": mi, "|相関|": pearson.to_numpy()}, index=mi_source)
compare.sort_values("相互情報量", ascending=False).round(3)


### 出力の読み方

`temperature_c`は**相互情報量は大きいのに|相関|は小さい**、という食い違いが見えるはずです。これが
「相関だけで特徴量を捨ててはいけない」理由です。両方を見て、関係の形は散布図で確かめます。


### 欠損の起こり方（欠損機構）を疑う

欠損はランダムとは限りません。**MCAR**（完全にランダム）、**MAR**（他の列で説明できる偏り）、
**MNAR**（値そのものに依存）で対処が変わります。ここでは「温度の欠損率が溶媒で偏るか」を見ます。


In [ ]:
miss = df.assign(temp_missing=df["temperature_c"].isna())
by_solvent = miss.groupby("solvent", dropna=False)["temp_missing"].mean().round(3)
print("溶媒別の温度欠損率:")
print(by_solvent)
print("溶媒でほぼ一定ならMCARに近い。偏るならMARを疑う。")


### 出力の読み方

溶媒によって温度欠損率が大きく違えば、欠損は溶媒と関係している（MARの疑い）＝
「一律に中央値で埋める」のが危ういサインです。値がほぼ一定なら、単純な補完でも大きな害は出にくいと判断できます。


### 多変量外れ値：組み合わせの異常を探す

「温度は普通、時間も普通、でもその組み合わせは他にない」という試料は、1列ずつ見ても見つかりません。
`IsolationForest`は**複数列を同時に見て、周囲から孤立した点**を外れ値候補として検出します。


In [ ]:
from sklearn.ensemble import IsolationForest

iso_cols = ["temperature_c", "reaction_time_h", "concentration_m", "yield_pct"]
iso_data = df[iso_cols].fillna(df[iso_cols].median())
flags = IsolationForest(contamination=0.03, random_state=42).fit_predict(iso_data)
outliers = df.loc[flags == -1, ["sample_id", *iso_cols]]
print("多変量外れ値候補:", len(outliers), "件")
outliers.round(2)


### 出力の読み方

- `contamination=0.03`は「全体の約3%を外れ値候補とみなす」設定です（多すぎ・少なすぎると感じたら調整）。
- 出た試料を1件ずつ見て、**どの列の組み合わせが変か**を考えます。単変量の箱ひげ図では正常だった試料が混じっていれば、多変量で見る価値があった、ということです。
- ここでも自動削除はせず、確認対象のリストとして扱います。


## APPENDIX（任意・追加演習）

可視化の引き出しを増やします。90分の外の自習向けです。まず**pairplot**で、複数の数値列の関係を
一度に俯瞰します（散布図と分布のマトリクス）。


In [ ]:
subset = df[["temperature_c", "reaction_time_h", "yield_pct", "catalyst"]].dropna()
sns.pairplot(subset, hue="catalyst", corner=True, plot_kws={"alpha": 0.5})


### 出力の読み方

対角線は各列の分布、対角線以外は2列の散布図で、色は触媒。**触媒ごとにかたまりができている**列の組が
あれば、それが効いている手がかり。多くの列を一気に眺めて当たりを付け、気になったペアを個別の図で
深掘りします（俯瞰→詳細の順）。


### バイオリン図とカウント図

**バイオリン図**は箱ひげ図より分布の形（山が1つか2つか）が分かります。**カウント図**はカテゴリの件数。
分布の形と件数を押さえると、平均の解釈がぐっと安全になります。


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.violinplot(data=df, x="catalyst", y="yield_pct", ax=axes[0])
axes[0].set_title("触媒別の収率分布（バイオリン）")
sns.countplot(data=df, x="solvent", ax=axes[1])
axes[1].set_title("溶媒の件数")
plt.tight_layout()


### 出力の読み方

- **バイオリン**：横幅が太い高さに値が集まっています。二山（2つのふくらみ）なら、隠れた別グループの存在を疑います。
- **カウント図**：件数の少ない溶媒は、以降の群別集計で平均が不安定になりやすい箇所。分析前に把握しておきます。


### 群別の目的変数と、目的変数との関連を棒で見る

「触媒別の活性率」と「収率との|相関|が強い列」を棒グラフで並べます。EDAの締めとして、
**目的変数（active/yield）に効きそうな列**の当たりを付けます。


In [ ]:
active_rate = df.groupby("catalyst")["active"].mean().sort_values(ascending=False)
corr_target = df.select_dtypes("number").corr()["yield_pct"].drop("yield_pct").abs().sort_values(ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
active_rate.plot.bar(ax=axes[0], title="触媒別の活性率")
corr_target.plot.bar(ax=axes[1], title="収率との|相関|")
plt.tight_layout()


### 出力の読み方

- **左**：触媒によって活性率が違えば、触媒は分類（active）に効く候補。
- **右**：収率との|相関|が高い列が、回帰（yield）で効く候補。ただし第4回本編のとおり、相関が低くても相互情報量が高い列（温度など）を見落とさないよう、相関の棒だけで判断しないこと。
- ここで挙がった候補列が、第5回以降の特徴量選びの出発点になります。


---

# パート2：何を、いつ、何のために予測するか

**このパートの問い：モデル構築より前に決めるべきことは何か。**


## 「良いモデル」の前に「正しい問い」

初心者がいちばん飛ばしがちで、実は最も効くのがこの回です。**どんなに精度が高くても、問いの立て方が
間違っていれば役に立ちません**。モデルを組む前に、次を1文で言えるようにします。

> **誰が・いつ・何を予測し・その結果をどう使うか。**

例：*実験条件を決める時点で使える情報から収率を予測し、優先して試す条件を選ぶ。*

ここで決定的に大事なのが**予測時点**です。「いつ予測するか」を決めると、その時点で**まだ手に入って
いない情報は使えない**と分かります。実験後にしか得られない値を入力に混ぜると、練習では高得点でも
本番でまったく使えない「ズル（リーク）」になります。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


## 計画時に使える列／使えない列を仕分ける

このデータで「実験条件を決める時点」を予測時点とすると、収率・活性・純度・測定後シグナルは
**まだ存在しません**。使える列と使えない列を、はっきり2つのリストに分けます。この仕分けが
特徴量選びの土台になります。


In [ ]:
available_at_planning = [
    "scaffold_group", "solvent", "catalyst", "temperature_c", "reaction_time_h",
    "concentration_m", "molecular_weight", "logp", "tpsa", "h_bond_donors", "rotatable_bonds",
]
unavailable_at_planning = ["yield_pct", "active", "post_assay_signal", "purity_pct"]
print("計画時に使える列:", available_at_planning)
print("実験後に得られる列:", unavailable_at_planning)


### 読みどころ

`unavailable_at_planning`の列は「結果」や「結果に強く連動する測定値」です。これらを特徴量に入れると
リークになります。**列の名前ではなく「その値がいつ確定するか」で判断する**のがコツです。


## TRY：まず「単純な基準（ベースライン）」を作る

複雑なモデルに進む前に、**平均値だけ／多数派だけ**を答える最も単純なモデルを作ります。これが
比較の出発点（ものさし）になります。以降のどのモデルも、まずこれを超えることが最低条件です。


In [ ]:
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.metrics import mean_absolute_error, accuracy_score
from sklearn.model_selection import train_test_split

train, valid = train_test_split(df, test_size=0.25, random_state=42)
reg = DummyRegressor(strategy="mean").fit(train[["molecular_weight"]], train["yield_pct"])
cls = DummyClassifier(strategy="most_frequent").fit(train[["molecular_weight"]], train["active"])
print("平均収率だけのMAE:", round(mean_absolute_error(valid["yield_pct"], reg.predict(valid[["molecular_weight"]])), 2))
print("多数派だけの正解率:", round(accuracy_score(valid["active"], cls.predict(valid[["molecular_weight"]])), 3))


### 出力の読み方

- **MAE（平均絶対誤差）**：予測が平均どれだけ外れるか。単位は収率と同じ%。「平均値だけ」でこの誤差、というものさしです。
- **多数派だけの正解率**が高く出ることに驚くかもしれません。活性が少ないデータでは「全部を多数派と答える」だけで正解率が高くなります。**だから正解率は当てにならない**——第8回でF1を学ぶ動機になります。
- 本命モデルは、この2つの数字を**はっきり上回って初めて価値がある**と考えます。
- なお`Dummy`は答え(`y`)だけを見て予測するため、ここで渡している`molecular_weight`列の中身は使いません（形式的な引数です）。


## CORE深掘り：リーク候補を自動で洗い出す

「どの列がリークか」を人手で全部見るのは大変です。目的変数と**極端に強く連動する列**は、結果由来の
情報が紛れている疑いがあります。それを見つける監査を関数にしておくと、自社データでも使い回せます。


In [ ]:
def leakage_audit(frame, target: str, threshold: float = 0.9):
    "目的変数と相関が極端に高い数値列を、リーク候補として洗い出す。"
    numeric = frame.select_dtypes(include="number")
    corr = numeric.corrwith(frame[target]).abs().drop(labels=[target], errors="ignore")
    report = corr.sort_values(ascending=False).to_frame("|相関|")
    report["リーク候補"] = report["|相関|"] >= threshold
    return report.round(3)

display(leakage_audit(df, target="active", threshold=0.6))
print("post_assay_signalは測定後の値。相関が高くても計画時には使えない。")


### 出力の読み方と注意

- `active`との|相関|が高い順に並びます。`post_assay_signal`が上位に来るはず——これは**活性測定後の値**なので、計画時には存在せず、使えばリークです。
- ただしこの監査は**あくまで補助**。相関が低くてもリークする列（例：実験日から結果を推測できる場合）もあります。最終判断は「その値がいつ確定するか」で人が行います。
- `threshold`はリーク候補とみなす相関の閾値。厳しく見たいなら下げます。


## TRY：自分のテーマを1枚に整理する

次の8点を、機密を書かずに埋めます。**利用者／判断／予測時点／目的変数／使える列／使えない列／
回帰か分類か／単純な基準**。埋まらない項目があれば、それが今いちばん詰めるべき点です。

## ASK COPILOT

Copilotには、曖昧な項目を勝手に埋めさせず「確認すべき質問」の形で返すよう頼みます。


## DEEP DIVE：最適な閾値は「コスト」で決まる

分類モデルは確率を出し、ある**閾値**を超えたら「活性」と判定します。既定の0.5が最適とは限りません。
最適な閾値は指標ではなく、**誤りのコスト**で決まります。ここでは「見逃し（偽陰性）＝有望条件を逃す損失」が
「偽陽性＝無駄な追試」より10倍高い状況を想定し、期待コストが最小になる閾値を探します。


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

feat = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa"]
X_tr, X_te, y_tr, y_te = train_test_split(df[feat], df["active"], test_size=0.3, random_state=42, stratify=df["active"])
clf = make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)).fit(X_tr, y_tr)
proba = clf.predict_proba(X_te)[:, 1]

cost_fn, cost_fp = 10, 1  # 見逃し=有望条件を逃す損失、偽陽性=無駄な追試
rows = []
for t in np.linspace(0.1, 0.9, 17):
    pred = (proba >= t).astype(int)
    fp = int(((pred == 1) & (y_te == 0)).sum())
    fn = int(((pred == 0) & (y_te == 1)).sum())
    rows.append({"閾値": round(t, 2), "偽陽性": fp, "偽陰性": fn, "期待コスト": fp * cost_fp + fn * cost_fn})
cost_table = pd.DataFrame(rows)
best = cost_table.loc[cost_table["期待コスト"].idxmin(), "閾値"]
display(cost_table)
print("コスト最小の閾値:", best, " / 見逃しが高いほど閾値は下がる")


### 出力の読み方

- 閾値を下げると「活性」と判定する数が増え、**偽陰性（見逃し）は減るが偽陽性は増える**トレードオフが表で見えます。
- 見逃しのコストが高いので、**最適閾値は0.5より低め**に出るはずです。「とりあえず0.5」がいかに恣意的かが分かります。
- コストの比（10:1）を変えれば最適閾値も動きます。**閾値はモデルの外側で、目的に合わせて選ぶ**ものだと理解できます。


### まとめ：指標は意思決定から逆算する

見逃しを避けたい探索段階なら**recall**寄り、追試コストが高い絞り込み段階なら**precision**寄り。
「良いスコア」を追うのではなく、「この予測で何を決め、間違えると何を失うか」から指標と閾値を選びます。
指標や閾値を選ぶときは、常にこの「何を決め、何を失うか」に立ち返ります。


## APPENDIX（任意・追加演習）

問題設定まわりのコードをもう少し。90分の外の自習向けです。まず**単純基準（Dummy）を戦略ごとに
比較**し、「どのベースラインを土俵にするか」を意識します。


In [ ]:
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.metrics import mean_absolute_error, accuracy_score
from sklearn.model_selection import train_test_split

tr, va = train_test_split(df, test_size=0.25, random_state=42)
print("=== 回帰の単純基準 ===")
for strat in ["mean", "median"]:
    d = DummyRegressor(strategy=strat).fit(tr[["molecular_weight"]], tr["yield_pct"])
    print(f"{strat:14s} MAE={mean_absolute_error(va['yield_pct'], d.predict(va[['molecular_weight']])):.2f}")
print("=== 分類の単純基準 ===")
for strat in ["most_frequent", "stratified", "uniform"]:
    d = DummyClassifier(strategy=strat, random_state=42).fit(tr[["molecular_weight"]], tr["active"])
    print(f"{strat:14s} accuracy={accuracy_score(va['active'], d.predict(va[['molecular_weight']])):.3f}")


### 出力の読み方

- 回帰は`mean`と`median`でMAEが少し違います。分布が歪んでいると`median`が有利なことも。
- 分類の`most_frequent`は正解率が高く見えますが、これは第8回で学ぶ「不均衡の罠」。`stratified`/`uniform`はランダムに近い基準です。
- 本命モデルは、**これらのうち最も手強い基準**を超えて初めて価値があります。


### 予測時点チェックを関数にする

第5回本編の「使える列／使えない列」を、候補リストから自動で仕分ける関数にします。自社データでも
使い回せる、実務的な安全装置です。


In [ ]:
def check_feature_timing(candidate_features, available_now, target):
    "特徴量候補を『使える/使えない(リーク)』に仕分ける。"
    rows = []
    for col in candidate_features:
        if col == target:
            verdict = "目的変数(使わない)"
        elif col in available_now:
            verdict = "使える"
        else:
            verdict = "使えない(予測時点で未確定)"
        rows.append({"列": col, "判定": verdict})
    return pd.DataFrame(rows)

check_feature_timing(
    ["temperature_c", "logp", "yield_pct", "post_assay_signal", "active"],
    available_now=available_at_planning,
    target="active",
)


### 出力の読み方

`yield_pct`や`post_assay_signal`が「使えない(予測時点で未確定)」と仕分けられます。列名を眺めるだけでなく、
**このチェックを通してから特徴量を確定する**運用にすれば、リークの多くを機械的に防げます。


### 期待値で「試すか否か」を決める

活性確率を予測できたとして、「その条件を追試すべきか」を**期待利益**で判断する簡単な例です。
確率×利益からコストを引いて、プラスなら試す。第5回・第8回のコスト最適閾値の考え方の土台です。


In [ ]:
import numpy as np

proba = np.array([0.10, 0.40, 0.60, 0.85])   # 各条件の活性確率（仮）
gain_if_active, cost_of_test = 100, 20
table = pd.DataFrame({"活性確率": proba})
table["期待利益"] = proba * gain_if_active - cost_of_test
table["試す?"] = table["期待利益"] > 0
table.round(1)


### 出力の読み方

期待利益がプラスの条件だけ「試す?=True」になります。ここでは損益分岐の確率は`cost/gain=0.2`。
つまり**活性確率20%以上なら試す**が最適で、これがそのまま判定閾値になります。「閾値0.5」が絶対でない
理由が、利益の式から自然に出てくることを確認してください。


---

# パート3：モデルは本当に当たっているか

**このパートの問い：手元のスコアをどこまで信じてよいか。**


## 「手元のスコア」を疑えるようになる回

前の回で「ベースラインより高いか」を見ました。でも、その高いスコアは**信じてよいのか**？
この回のテーマはそこです。データサイエンスで最も高くつく失敗は、計算ミスではなく
**「当たっているつもり」で外すこと**。原因はたいてい次の2つです。

- **過学習**：学習データを覚えすぎ、未知データに弱い。
- **リーク（データ漏れ）**：予測時に手に入らない情報が学習に混ざり、練習だけ高得点になる。

まず、学習と検証を分けたデータを用意します（`dropna`で欠損行を落として単純化）。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

features = ["molecular_weight", "logp", "tpsa", "temperature_c", "reaction_time_h"]
clean = df.dropna(subset=features)
X_train, X_valid, y_train, y_valid = train_test_split(clean[features], clean["active"], test_size=0.25, random_state=42, stratify=clean["active"])


## TRY：木を深くすると「過学習」が見える

決定木の深さ（`max_depth`）を段階的に深くして、**学習データでの正解率**と**検証データでの正解率**を
並べます。深くするほど学習側は上がりますが、検証側はどこかで頭打ち・悪化します。この2つの差が
「覚えすぎ」の度合いです。


In [ ]:
rows = []
for depth in [1, 2, 4, 8, None]:
    model = DecisionTreeClassifier(max_depth=depth, random_state=42).fit(X_train, y_train)
    rows.append({
        "max_depth": str(depth),
        "学習スコア": accuracy_score(y_train, model.predict(X_train)),
        "検証スコア": accuracy_score(y_valid, model.predict(X_valid)),
    })
pd.DataFrame(rows).round(3)


### 出力の読み方

- `max_depth=None`（無制限）では**学習スコアが1.0近く**まで上がるのに、**検証スコアはそれほど伸びない**——典型的な過学習です。
- 検証スコアが最も高い深さの手前あたりが「ちょうど良い複雑さ」。「学習スコアの高さ」を実力だと勘違いしないことが、ここでの分かれ目です。
- 教訓：**必ず「未知データ役（検証）」で評価する**。学習データでの高得点は実力ではありません。


## TRY：リーク列を入れると「不自然に」良くなる

わざと`post_assay_signal`（活性測定後の値）を特徴量に混ぜてみます。予測したい`active`と強く連動する
ため、検証スコアが不自然に跳ね上がります。**練習では高得点なのに本番で使えない**典型です。


In [ ]:
leak_features = [*features, "post_assay_signal"]
leaked = df.dropna(subset=leak_features)
Xl_tr, Xl_va, yl_tr, yl_va = train_test_split(leaked[leak_features], leaked["active"], test_size=0.25, random_state=42, stratify=leaked["active"])
leaked_model = DecisionTreeClassifier(max_depth=3, random_state=42).fit(Xl_tr, yl_tr)
print("リーク列ありの検証スコア:", round(accuracy_score(yl_va, leaked_model.predict(Xl_va)), 3))
print("post_assay_signalは測定後の値。計画時の予測には使えません。")


### 出力の読み方

リーク列を入れた検証スコアは、リークなしのときより**明らかに高い**はずです。**「スコアが急に良くなったら喜ぶ前に疑う」**——高すぎるスコアはリークの最初のサインです。第5回のリーク監査と合わせて習慣にします。


## CORE深掘り：前処理も「分割の内側」で行う

見落としやすいリークが**前処理リーク**です。標準化や欠損補完を**全データで先に**行うと、検証データの
情報（平均など）が学習へこっそり混ざります。正しくは、前処理も交差検証の**各分割の内側**で学習します。
`Pipeline`に前処理を入れると、これが自動で守られます。


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

filled = clean[features].fillna(clean[features].median())
scaler_all = StandardScaler().fit(filled)               # 誤り：全データで学習
leaked_scores = cross_val_score(LogisticRegression(max_iter=1000), scaler_all.transform(filled), clean["active"], cv=5, scoring="f1")

right_pipe = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=1000))
right_scores = cross_val_score(right_pipe, clean[features], clean["active"], cv=5, scoring="f1")
print("全データ前処理(楽観的) F1平均:", round(leaked_scores.mean(), 3))
print("Pipeline内前処理(正しい) F1平均:", round(right_scores.mean(), 3))


### 出力の読み方

このデータでは差は小さいかもしれませんが、ここで確かめたいのは**やり方が正しいかどうか**そのものです。全データで前処理する方式は
原理的に楽観へ偏ります。`Pipeline`にまとめれば、分割ごとに前処理を学習し直すので安全——だから第9回で
`Pipeline`を本格的に学びます。


## CHALLENGE：似た試料を「両側に入れない」分割

同じ化合物系列（scaffold）の似た分子が学習側と検証側の両方に入ると、検証が甘くなります（実質カンニング）。
`GroupShuffleSplit`で**系列ごとまるごと**どちらかへ振り分けると、より本番に近い評価になります。


In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, valid_idx = next(splitter.split(clean, groups=clean["scaffold_group"]))
print("学習側の系列:", sorted(clean.iloc[train_idx]["scaffold_group"].unique()))
print("検証側の系列:", sorted(clean.iloc[valid_idx]["scaffold_group"].unique()))


### 出力の読み方

学習側と検証側で**系列(scaffold_group)が重ならない**ことを確認します。新規骨格への予測力を測りたいなら、
この「群を跨がせない分割」が正しい評価です。ランダム分割より点数は下がりがちですが、それが**本当の実力**です。


## CORE深掘り：学習・検証・テストの3つに分ける

ここまでは学習用と検証用の2つでした。実務では**3つ**に分けます。**検証(valid)は設定選びに何度でも使い**、
**テスト(test)は最後の1回だけ**触ります。何度も見た検証データには無意識に合わせ込んでしまうため、
「一度も見ていないテスト」で最終性能を確かめる、という役割分担です。


In [ ]:
from sklearn.model_selection import train_test_split

# まずテストを切り分け（最後まで触らない）、残りを学習用と検証用へ
work, test_set = train_test_split(clean, test_size=0.2, random_state=42, stratify=clean["active"])
train_set, valid_set = train_test_split(work, test_size=0.25, random_state=42, stratify=work["active"])
print("学習用:", len(train_set), "件（モデルを学習）")
print("検証用:", len(valid_set), "件（設定選び・改善判断に何度でも使う）")
print("テスト用:", len(test_set), "件（最後の確認まで開かない）")


### 出力の読み方

3つの件数が表示されます。**検証とテストの違い**はサイズではなく**使い方**です。検証は改善のたびに何度でも
見てよい／テストは最後に1回だけ。この分担を守ると、「検証データに合わせ込んで実力を過大評価する」失敗を
防げます（この回の振り返り「検証とテストの違いは何か」は、このセルを指させればOKです）。


## DEEP DIVE：分割方式で「楽観度」はこんなに変わる

評価とは「将来の使われ方を模擬すること」。だから分割方式の選択が結果を左右します。同じモデルを
3つの分割方式（ふつうのKFold／層化／系列で分けるGroup）で評価し、スコアがどう変わるかを見ます。


In [ ]:
from sklearn.model_selection import KFold, StratifiedKFold, GroupKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier

estimator = make_pipeline(SimpleImputer(strategy="median"), DecisionTreeClassifier(max_depth=4, random_state=42))
X_all, y_all, groups = df[features], df["active"], df["scaffold_group"]
schemes = {
    "KFold": cross_val_score(estimator, X_all, y_all, cv=KFold(5, shuffle=True, random_state=42), scoring="f1"),
    "StratifiedKFold": cross_val_score(estimator, X_all, y_all, cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring="f1"),
    "GroupKFold(系列)": cross_val_score(estimator, X_all, y_all, cv=GroupKFold(5), groups=groups, scoring="f1"),
}
pd.DataFrame({name: {"平均": s.mean(), "標準偏差": s.std(), "最低": s.min()} for name, s in schemes.items()}).T.round(3)


### 出力の読み方

- **GroupKFold（系列）の平均が最も低く**出るのが普通です。似た試料を跨がせないぶん厳しく、これが新規骨格への実力に近い。
- 「どの分割が正しいか」は**将来の使い方**で決まります。新しい系列に使うならGroup、同じ系列内での予測ならKFoldでも可。
- 標準偏差（ばらつき）も見て、平均だけで判断しません。


### ネストCV：設定選びと性能報告を分ける

`max_depth`などの設定を「検証スコアが最高になるよう」選び、その同じ検証スコアを性能として報告すると、
**出来すぎの数字**になります。これを防ぐのがネストCV：**内側のCVで設定を選び、外側のCVで評価**します。


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier

pipe = make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(random_state=42))
param_dist = {
    "randomforestclassifier__n_estimators": [100, 200, 300],
    "randomforestclassifier__max_depth": [3, 4, 6, None],
    "randomforestclassifier__min_samples_leaf": [1, 2, 4],
}
inner = StratifiedKFold(3, shuffle=True, random_state=1)
outer = StratifiedKFold(5, shuffle=True, random_state=2)
search = RandomizedSearchCV(pipe, param_dist, n_iter=8, cv=inner, scoring="f1", random_state=42)
nested = cross_val_score(search, df[features], df["active"], cv=outer, scoring="f1")
print("ネストCVの外側F1:", nested.round(3))
print("楽観の少ない推定 平均±SD:", round(nested.mean(), 3), "±", round(nested.std(), 3))


### 出力の読み方

外側5分割それぞれで「内側で設定を選び直し→未見の外側で評価」しています。ここで出る平均が、
**設定選びの下駄を履いていない、より正直な性能**です。単純なグリッド探索の最高スコアより低めに
出るのが健全で、その差が「探索による楽観」の大きさです。


### adversarial validation：学習とテストは似ているか

もう1つの落とし穴が**分布ずれ**（学習データとテストデータの傾向が違う）です。「その行が学習か
テストか」を当てる分類器を作り、そのAUC（当てやすさ）で分布の近さを測ります。


In [ ]:
train_c = pd.read_csv(DATA / "local_competition" / "train.csv")
test_c = pd.read_csv(DATA / "local_competition" / "test.csv")
adv_features = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa"]
combined = pd.concat([
    train_c[adv_features].assign(is_test=0),
    test_c[adv_features].assign(is_test=1),
], ignore_index=True)
adv_model = make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=200, random_state=42))
auc = cross_val_score(adv_model, combined[adv_features], combined["is_test"], cv=5, scoring="roc_auc")
print("adversarial validation AUC:", round(auc.mean(), 3))
print("0.5付近なら分布は近い。0.8以上なら分布ずれを疑う。")


### 出力の読み方

- **AUC≈0.5**：学習とテストが見分けられない＝分布が近い。手元のCVは信頼できます。
- **AUC≫0.5（0.8以上など）**：見分けがつく＝分布がずれており、手元のCVは本番を過大評価しがち。
- このデータは同じ生成過程なのでAUCは0.5付近のはず。実データでこの値が高ければ、時系列や機器差など「ずれの原因」を探します。


## APPENDIX（任意・追加演習）

交差検証の中身を、あえて手作りして仕組みを体で理解します。90分の外の自習向けです。
`cross_val_score`が内部でやっていることを、`for`ループで書き下します。


In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score

data = df.dropna(subset=features).reset_index(drop=True)
Xa, ya = data[features], data["active"]
k = 5
fold_id = np.arange(len(Xa)) % k          # 位置でfoldを割り当てる（デモ用）
scores = []
for f in range(k):
    is_valid = fold_id == f
    m = DecisionTreeClassifier(max_depth=4, random_state=42).fit(Xa[~is_valid], ya[~is_valid])
    scores.append(f1_score(ya[is_valid], m.predict(Xa[is_valid])))
print("手作りk-fold F1:", [round(s, 3) for s in scores])
print("平均:", round(np.mean(scores), 3))


### 出力の読み方

「4つのfoldで学習→残り1つで検証」を5回繰り返し、平均しています。これが`cross_val_score`の正体です。
中身が分かると、**分割の乱数や層化（stratify）を変えると平均が動く**ことも納得できます。


### 時系列分割：未来で過去を検証しない

`experiment_date`で並べ、`TimeSeriesSplit`で「過去で学習→未来で検証」を繰り返します。実運用が
「過去データで学習し、これから来る試料を予測する」形なら、この分割が最も現実に近い評価です。


In [ ]:
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer

time_sorted = df.sort_values("experiment_date")
est = make_pipeline(SimpleImputer(strategy="median"), DecisionTreeClassifier(max_depth=4, random_state=42))
tscv = TimeSeriesSplit(n_splits=5)
ts_scores = cross_val_score(est, time_sorted[features], time_sorted["active"], cv=tscv, scoring="f1")
print("時系列分割F1:", ts_scores.round(3), " 平均:", round(ts_scores.mean(), 3))


### 出力の読み方

各foldは「それまでの期間」で学習し「直後の期間」で検証します。前半のfoldは学習データが少なく不安定に
なりがち。時間で性能が変わるなら、ランダム分割より厳しい（現実的な）数字が出ます。


### shuffleの有無で結果は変わる

`KFold`の`shuffle`を切り替えて比較します。データが何らかの順序（日付・バッチ順など）で並んでいると、
`shuffle=False`は偏った分割になり、スコアが不安定・楽観/悲観に振れることがあります。


In [ ]:
from sklearn.model_selection import KFold

for shuffle in [False, True]:
    kf = KFold(5, shuffle=shuffle, random_state=42 if shuffle else None)
    s = cross_val_score(est, df[features], df["active"], cv=kf, scoring="f1")
    print(f"shuffle={str(shuffle):5s}: {s.round(3)}  平均={s.mean():.3f}")


### 出力の読み方

2つの平均やばらつきが違えば、**データの並び順が結果に影響している**証拠。ふつうは`shuffle=True`が無難ですが、
時系列データでは`shuffle`してはいけません（未来が学習に混ざる）。「どう並んでいるか」を意識して分割を選びます。


---

## よくある誤り

- 外れ値を自動削除する
- 相関を因果と読む
- 見栄えの良い図だけを選ぶ
- 入手できる列をすべて使う
- 目的変数が測定や運用で不安定
- 精度目標だけで利用方法とコストが決まっていない
- 前処理を全データで済ませてから分割する
- 同じ系列の類似化合物を両側へ入れる
- 検証データを何度も見て実質的に学習する

## SELF-STUDY（任意・30〜60分）

- 相互情報量の上位3列について、散布図で関係の形を確認する
- IsolationForestの外れ値候補2件を、残す場合と除く場合で整理する
- 自社テーマを機密情報なしで問題設定キャンバスへ落とす
- 偽陽性・偽陰性のコストを入れ、期待コスト最小の閾値を計算する
- KFold・StratifiedKFold・GroupKFoldのF1分布を箱ひげ図で比べる
- adversarial validationのAUCを下げる列を1つ見つけ理由を書く

成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

## 振り返りチェック

1. 相関係数と相互情報量はどう違うか
2. MCARとMARの違いは何か
3. 多変量外れ値が単変量で見つからない理由は何か
4. 誰が何を判断するモデルか
5. 予測時点で本当に得られる列はどれか
6. コスト行列から最適な閾値をどう求めるか
7. 検証とテストの違いは何か
8. ネストCVが必要になるのはどんなときか
9. adversarial validationのAUCが高いと何を意味するか

答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
